# P0：RQ1 机场延误净变化（原始描述）

研究问题：哪些机场的同机相邻航段衔接平均吸收或放大延误？本阶段先报告开发期原始汇总；共同支持标准化、日期块区间和阈值敏感性留待 P5。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "eda_summary.json").exists()
                  and (p / "rq1_summary.json").exists()]
    if not candidates:
        raise RuntimeError("请先运行 P0 开发分析，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

## 1. 主样本与三个不同的量

主样本来自有效衔接，要求上游非负到达延误至少 15 分钟、计划非隔夜。RQ1 允许实际地面时间为 0；不套用 RQ2 的严格预测早于出发条件。开发期先按上游实际到达 UTC 4–7 月筛选，再分析目标。

\[\Delta_i = D_{down,i}^{dep} - D_{up,i}^{arr}\]

**每次平均净变化**的单位是分钟/次衔接；**累计增加** \(S_+=\sum \max(\Delta_i,0)\)、**累计减少** \(S_-=\sum\max(-\Delta_i,0)\) 和 **累计净变化** \(S_{net}=\sum\Delta_i=S_+-S_-\) 的单位是样本总分钟。累计量受样本量影响，不能直接当作机场表现。

In [ ]:
summary = receipt("rq1_summary.json")
display(pd.DataFrame(list(summary["counts"].items()), columns=["顺序样本流", "数量"]))
display(pd.DataFrame([{k: v for k, v in summary.items() if k not in ["counts", "interpretation"]}]))
assert summary["delta_identity_checked"]

## 2. 机场原始均值、分布与支持

按样本量选择最大的 20 个机场展示；图内顺序不构成标准化排名。表中 mean/median/p10/p90、减少/增加/不变比例均使用同一主样本。完整机场表归档于 tables/rq1_airport_raw.csv。

In [ ]:
raw = table("rq1_airport_raw.csv")
display(raw.head(20))
figure("rq1_raw_airport_mean.png")

## 3. 累计增加、累计减少与净值

较大机场的累计分钟可能很高，即使其每次平均净变化较小。必须将累计量与样本量及每次平均量一起阅读。逐连接与逐机场均验证 Snet = S+ − S−，避免正负变化混算。

In [ ]:
figure("rq1_cumulative_delay.png")
display(raw[["airport_id", "airport_code", "n", "mean_delta", "s_plus", "s_minus", "s_net"]].head(20))

## 4. 分月支持及上游延误 × 计划周转

逐月表帮助识别月份组成变化。热图分别列计数与平均净变化：低支持格不足以支持稳定结论。没有正式最低周转时间，本项目只使用计划周转，不把它直接命名为可吸收缓冲。

In [ ]:
monthly = table("rq1_airport_monthly.csv")
display(monthly.groupby("month_utc")["n"].sum().rename("connections").to_frame())
display(monthly.head(20))
figure("rq1_turnaround_count.png")
figure("rq1_turnaround_mean_delta.png")

## 5. 当前可以与尚不能得出的结论

负 Δ 表示这次衔接保留的下游延误少于上游晚到；下游仍可能延误。当前机场差异尚未按共同上游延误和计划周转分布标准化，也没有抽样不确定性区间，因此只能描述本样本的原始表现，不能解释为机场因果责任或正式优劣排名。